In [ ]:
%pip uninstall --yes 'keras' 'matplotlib' 'scikit-learn' 'tensorflow'

In [ ]:
import warnings
warnings.simplefilter('ignore')

In [ ]:
import os
import sys
import subprocess

In [ ]:
def set_env(input_archive, temp_dir):

    if not os.path.exists(temp_dir):
        os.makedirs(temp_dir, exist_ok=True)
        
        subprocess.run(['tar', '-xzf', input_archive, '-C', temp_dir], check=True)
    
    subprocess.run([
        sys.executable, 
        '-m', 
        'pip', 
        'install', 
        '--no-index', 
        '--find-links', 
        f'{temp_dir}/wheels', 
        'unsloth', 
        'trl', 
        'vllm', 
        'openai_harmony'
    ], check=True)

In [ ]:
set_env(
    input_archive='/kaggle/input/aimo-3-utils/wheels.tar.gz', 
    temp_dir='/kaggle/tmp/setup'
)

In [ ]:
subprocess.run(['ls', '/kaggle/tmp/setup/tiktoken_encodings'])


def find_model_path(model_name: str = 'gpt-oss-120b') -> str:
    """Auto-discover model path from /kaggle/input/."""
    import glob

    candidates = [
        # model_sources mount pattern (new Kaggle)
        f'/kaggle/input/models/danielhanchen/{model_name}/transformers/default/1',
        f'/kaggle/input/models/openai/{model_name}/transformers/default/1',
        # dataset-style mount pattern (old Kaggle / some configs)
        f'/kaggle/input/{model_name}/transformers/default/1',
        f'/kaggle/input/{model_name}',
    ]

    for path in candidates:
        if os.path.isfile(os.path.join(path, 'config.json')):
            print(f'Model found at: {path}')
            return path

    # Fallback: search for any config.json with model weights
    for config_path in sorted(glob.glob('/kaggle/input/**/config.json', recursive=True)):
        parent = os.path.dirname(config_path)
        if any(glob.glob(os.path.join(parent, f'*.{ext}')) for ext in ['safetensors', 'bin']):
            print(f'Model found via search: {parent}')
            return parent

    # Debug dump
    print('ERROR: Model not found. /kaggle/input/ contents:')
    for root, dirs, files in os.walk('/kaggle/input/', topdown=True):
        depth = root.replace('/kaggle/input/', '').count(os.sep)
        if depth < 5:
            print(f'{"  " * depth}{os.path.basename(root)}/ ({len(files)} files)')
        if depth >= 5:
            dirs[:] = []

    raise RuntimeError(f'Model "{model_name}" not found under /kaggle/input/')


MODEL_PATH = find_model_path()
print(f'\nUsing model path: {MODEL_PATH}')

In [ ]:
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'
os.environ['TIKTOKEN_ENCODINGS_BASE'] = '/kaggle/tmp/setup/tiktoken_encodings'

In [ ]:
import gc
import re
import math
import time
import queue
import threading
import contextlib
from typing import Optional
from jupyter_client import KernelManager
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor

import pandas as pd
import polars as pl

from openai import OpenAI

from openai_harmony import (
    HarmonyEncodingName, 
    load_harmony_encoding, 
    SystemContent, 
    ReasoningEffort, 
    ToolNamespaceConfig, 
    Author, 
    Message, 
    Role, 
    TextContent, 
    Conversation
)

from transformers import set_seed
import kaggle_evaluation.aimo_3_inference_server

In [ ]:
class CFG:
    
    system_prompt = (
        'You are an elite mathematical problem solver with expertise at the International '
        'Mathematical Olympiad (IMO) level. Your goal is to find the correct answer through '
        'rigorous mathematical reasoning.\n\n'
        
        '# Problem-Solving Approach:\n'
        '1. UNDERSTAND: Carefully read and rephrase the problem in your own words. '
        'Identify what is given, what needs to be found, and any constraints.\n'
        '2. EXPLORE: Consider multiple solution strategies. Think about relevant theorems, '
        'techniques, patterns, or analogous problems. Don\'t commit to one approach immediately.\n'
        '3. PLAN: Select the most promising approach and outline key steps before executing.\n'
        '4. EXECUTE: Work through your solution methodically. Show all reasoning steps clearly.\n'
        '5. VERIFY: Check your answer by substituting back, testing edge cases, or using '
        'alternative methods. Ensure logical consistency throughout.\n\n'
        
        '# Mathematical Reasoning Principles:\n'
        '- Break complex problems into smaller, manageable sub-problems\n'
        '- Look for patterns, symmetries, and special cases that provide insight\n'
        '- Use concrete examples to build intuition before generalizing\n'
        '- Consider extreme cases and boundary conditions\n'
        '- If stuck, try working backwards from the desired result\n'
        '- Be willing to restart with a different approach if needed\n\n'
        
        '# Verification Requirements:\n'
        '- Cross-check arithmetic and algebraic manipulations\n'
        '- Verify that your solution satisfies all problem constraints\n'
        '- Test your answer with simple cases or special values when possible\n'
        '- Ensure dimensional consistency and reasonableness of the result\n\n'
        
        '# Output Format:\n'
        'The final answer must be a non-negative integer between 0 and 99999.\n'
        'When you compute a candidate answer through any approach, IMMEDIATELY save it as \\Vboxed{N} '
        '(e.g., \\Vboxed{42}). This is your checkpoint — continue verifying or trying other approaches. '
        'When you are confident, write your final answer as \\boxed{N}. '
        'If you find a better answer later, write a new \\Vboxed{N} to update your checkpoint.\n'
        'Always have at least one \\Vboxed{} before attempting verification.\n\n'
        
        '# Time Management:\n'
        'You have a strict deadline. A variable `_DEADLINE` is set in your Python environment.\n'
        'Check remaining time: `print(f"{_DEADLINE - _time.time():.0f}s left")`\n'
        'You will receive automatic warnings in code output when time is low:\n'
        '- ~100s remaining: time notice — be mindful of remaining time\n'
        '- ~30s remaining: you MUST write \\Vboxed{N} immediately before doing anything else\n'
        '- ~5s remaining: system takes over — code execution is blocked, answer is extracted automatically\n'
        'When you see a time warning, write \\Vboxed{N} BEFORE doing anything else.\n\n'
        
        'Think step-by-step and show your complete reasoning process. Quality of reasoning '
        'is as important as the final answer.\n\n'
        
        '# Strategic Rules:\n'
        '- If your computed answer is NOT a whole number in 0-99999 (negative, decimal, or too large), '
        'you have misinterpreted the problem. Re-read the question and try a different interpretation.\n'
        '- If your code consistently computes X but your reasoning suggests Y, trust the code. '
        'Empirical results from multiple independent runs outweigh theoretical arguments.\n'
        '- For combinatorial problems, verify your formula against brute-force for 3-5 small cases '
        'before extrapolating to the full problem size.\n'
        '- If your code execution keeps timing out, submit your best partial result '
        'rather than retrying indefinitely.\n'
        '- Before writing code, estimate computational complexity. If the search space exceeds '
        '10^6 operations, find a mathematical shortcut, recurrence, or modular arithmetic approach '
        'instead of brute force.\n'
        '- If code times out, do NOT just reduce the range and retry. Instead: '
        '(a) find a closed-form or recurrence, (b) use modular arithmetic to avoid large numbers, '
        '(c) reformulate as dynamic programming.\n'
        '- If you have 3+ timeouts in the first 5 code cells, STOP coding and reason through the '
        'problem mathematically. Your brute-force approach is failing — find the insight.\n\n'
        
        '# Efficiency:\n'
        'If the problem has an obvious, immediate answer (e.g. direct computation, '
        'well-known identity, or trivial formula application), state the answer '
        'directly with brief justification. Do not use Python for problems you can '
        'solve in 3 lines of reasoning.'
    )
    
    tool_prompt = (
        'Use this tool to execute Python code for:\n'
        '- Complex calculations that would be error-prone by hand\n'
        '- Numerical verification of analytical results\n'
        '- Generating examples or testing conjectures\n'
        '- Visualizing problem structure when helpful\n'
        '- Brute-force verification for small cases\n\n'
        
        'The environment is a stateful Jupyter notebook. Code persists between executions.\n'
        'Always use print() to display results. Write clear, well-commented code.\n\n'
        
        'Remember: Code should support your mathematical reasoning, not replace it. '
        'Explain what you\'re computing and why before running code.'
    )
    
    preference_prompt = (
        'You have access to `math`, `numpy`, and `sympy` for:\n\n'
        
        '# Symbolic Computation (sympy):\n'
        '- Algebraic manipulation and simplification\n'
        '- Solving equations and systems of equations\n'
        '- Symbolic differentiation and integration\n'
        '- Number theory functions (primes, divisors, modular arithmetic)\n'
        '- Polynomial operations and factorization\n'
        '- Working with mathematical expressions symbolically\n\n'
        
        '# Numerical Computation (numpy):\n'
        '- Array operations and linear algebra\n'
        '- Efficient numerical calculations for large datasets\n'
        '- Matrix operations and eigenvalue problems\n'
        '- Statistical computations\n\n'
        
        '# Mathematical Functions (math):\n'
        '- Standard mathematical functions (trig, log, exp)\n'
        '- Constants like pi and e\n'
        '- Basic operations for single values\n\n'
        
        'Best Practices:\n'
        '- Use sympy for exact symbolic answers when possible\n'
        '- Use numpy for numerical verification and large-scale computation\n'
        '- Combine symbolic and numerical approaches: derive symbolically, verify numerically\n'
        '- Document your computational strategy clearly\n'
        '- Validate computational results against known cases or theoretical bounds\n'
        '- For very large exponents (e.g. a^(n!)), use pow(base, exp, mod) or analytical methods — never materialize the full number\n\n'
        
        '# Code Robustness Rules:\n'
        '- Each code cell must be SELF-CONTAINED: re-import and re-define everything you need. '
        'If a previous cell errored, its definitions are LOST.\n'
        '- Avoid sympy expressions that grow beyond a few hundred characters. '
        'If an expression is exploding, switch to numerical methods (numpy) or modular arithmetic.\n'
        '- For geometry: prefer numpy coordinate models with brute-force verification '
        'over pure symbolic or direction-counting approaches.\n'
        '- For combinatorics counting: always verify no duplicates — check injectivity explicitly.\n'
        '- If 5+ code cells without progress, STOP and restart with a different approach.\n'
        '- When using Python, prefer converting sympy expressions to int() before passing to Python builtins like pow(), min(), max().\n'
        '- Common import: from sympy.ntheory.modular import crt (note: sympy.crt does not exist).\n'
        '- If a code cell fails, re-import necessary libraries in the next cell.\n\n'
        '- sympy.solve() often times out on complex systems. For polynomial systems with > 3 unknowns, '
        'prefer numerical methods (scipy fsolve/minimize) or manual algebraic reduction.\n'
        '- Never call sympy.simplify() or sympy.expand() on expressions involving large binomials or '
        'products of many terms. Use float() first to check the numerical value.\n'
        
        '# Library Availability:\n'
        'NOT installed (do NOT import — they WILL fail): pulp, ortools, z3-solver, mip, pyscipopt, pysat, cvxpy, constraint, matplotlib, sklearn.\n'
        'Do not waste turns trying these libraries or probing with try/except.\n\n'
        
        'Available and recommended:\n'
        '- sympy, numpy, scipy, networkx, mpmath, math, itertools, functools, fractions, collections, decimal, heapq, random, statistics\n\n'
        
        'Replacements for unavailable optimization/constraint libraries:\n'
        '- Integer/Binary Linear Programming (replaces pulp, mip, ortools):\n'
        '  from scipy.optimize import milp, LinearConstraint, Bounds\n'
        '  import numpy as np\n'
        '  res = milp(c=c_vec, constraints=LinearConstraint(A, lb, ub),\n'
        '             integrality=np.ones(n), bounds=Bounds(0, 1))\n'
        '  # res.success, res.x (solution vector), res.fun (objective value)\n'
        '- Linear Programming: scipy.optimize.linprog(c, A_ub=A, b_ub=b, method="highs")\n'
        '- Assignment problems: scipy.optimize.linear_sum_assignment(cost_matrix)\n'
        '- Graph algorithms (matching, shortest path, components): import networkx as nx\n'
        '- Constraint satisfaction: encode as ILP via scipy.optimize.milp, or backtracking with pruning\n'
        '- Number theory: sympy.ntheory or random (Monte Carlo verification)\n'
        '- Geometry: fractions (exact rational) + sympy, avoid floating point\n'
        '- Game theory/DP: functools.lru_cache for memoized recursion'
    )
    
    served_model_name = 'gpt-oss'
    model_path = MODEL_PATH
    
    kv_cache_dtype = 'fp8_e4m3'
    dtype = 'auto'

    problem_timeout = 900

    notebook_limit = 17400
    server_timeout = 180

    session_timeout = 960
    jupyter_timeout = 30
    sandbox_timeout = 3

    stream_interval = 200
    context_tokens = 65536
    buffer_tokens = 512
    search_tokens = 32
    top_logprobs = 5
    batch_size = 256
    attempts = 16
    workers = 16
    turns = 128
    seed = 42

    gpu_memory_utilization = 0.96
    temperature = 0.5
    temp_schedule = [0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5]
    min_p = 0.02


In [ ]:
set_seed(CFG.seed)

In [ ]:
class AIMO3Template:

    def __init__(self):

        pass

    def get_system_content(self, system_prompt: str, tool_config: ToolNamespaceConfig) -> SystemContent:

        return (
            SystemContent.new()
            .with_model_identity(system_prompt)
            .with_reasoning_effort(reasoning_effort=ReasoningEffort.HIGH)
            .with_tools(tool_config)
        )

    def apply_chat_template(
        self, 
        system_prompt: str, 
        user_prompt: str, 
        tool_config: ToolNamespaceConfig
    ) -> list[Message]:

        system_content = self.get_system_content(system_prompt, tool_config)        
        system_message = Message.from_role_and_content(Role.SYSTEM, system_content)

        user_message = Message.from_role_and_content(Role.USER, user_prompt)

        return [system_message, user_message]

In [ ]:
class AIMO3Sandbox:

    _port_lock = threading.Lock()
    _next_port = 50000

    @classmethod
    def _get_next_ports(cls, count: int = 5) -> list[int]:

        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count

            return ports

    def __init__(self, timeout: float):

        self._default_timeout = timeout
        self._owns_kernel = False
        self._client = None
        self._km = None
        
        self._start_kernel()

    def _start_kernel(self):
        """Start a fresh Jupyter kernel and initialize it."""
        ports = self._get_next_ports(5)

        env = os.environ.copy()
        env['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
        env['PYDEVD_WARN_EVALUATION_TIMEOUT'] = '0'
        env['JUPYTER_PLATFORM_DIRS'] = '1'
        env['PYTHONWARNINGS'] = 'ignore'
        env['MPLBACKEND'] = 'Agg'

        self._km = KernelManager()
        self._km.shell_port = ports[0]
        self._km.iopub_port = ports[1]
        self._km.stdin_port = ports[2]
        self._km.hb_port = ports[3]
        self._km.control_port = ports[4]

        self._km.start_kernel(env=env, extra_arguments=['--Application.log_level=CRITICAL'])

        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self._default_timeout)
        self._owns_kernel = True

        self._init_environment()

    def _init_environment(self):
        """Run the standard imports in the kernel."""
        self.execute(
            'import sys\n'
            'sys.set_int_max_str_digits(100000)\n'
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import functools\n'
            'import fractions\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
        )

    def _format_error(self, traceback: list[str]) -> str:

        clean_lines = []

        for frame in traceback:
            clean_frame = re.sub(r'\x1b\[[0-9;]*m', '', frame)

            if 'File "' in clean_frame and 'ipython-input' not in clean_frame:
                continue

            clean_lines.append(clean_frame)

        return ''.join(clean_lines)

    def execute(self, code: str, timeout: float | None = None) -> str:

        client = self._client
        effective_timeout = timeout or self._default_timeout
        
        msg_id = client.execute(
            code, 
            store_history=True, 
            allow_stdin=False, 
            stop_on_error=False
        )

        stdout_parts = []
        stderr_parts = []
        
        start_time = time.time()

        while True:
            elapsed = time.time() - start_time

            if elapsed > effective_timeout:
                self._km.interrupt_kernel()
                # Force-kill kernel to stop zombie C-level computations (e.g., HiGHS MIP solver)
                # interrupt_kernel() only sends SIGINT which C libraries may ignore
                self._force_restart()

                return f'[ERROR] Execution timed out after {effective_timeout} seconds'

            try:
                msg = client.get_iopub_msg(timeout=1.0)

            except queue.Empty:
                continue

            if msg.get('parent_header', {}).get('msg_id') != msg_id:
                continue

            msg_type = msg.get('msg_type')
            content = msg.get('content', {})

            if msg_type == 'stream':
                text = content.get('text', '')

                if content.get('name') == 'stdout':
                    stdout_parts.append(text)

                else:
                    stderr_parts.append(text)

            elif msg_type == 'error':
                traceback_list = content.get('traceback', [])

                stderr_parts.append(self._format_error(traceback_list))

            elif msg_type in {'execute_result', 'display_data'}:
                data = content.get('data', {})
                text = data.get('text/plain')

                if text:
                    stdout_parts.append(text if text.endswith('\n') else f'{text}\n')

            elif msg_type == 'status':
                if content.get('execution_state') == 'idle':
                    break

        stdout = ''.join(stdout_parts)
        stderr = ''.join(stderr_parts)

        if stderr:
            return f'{stdout.rstrip()}\n{stderr}' if stdout else stderr

        return stdout if stdout.strip() else '[WARN] No output. Use print() to see results.'

    def _force_restart(self):
        """Kill the current kernel and start a fresh one. Handles zombie C-level computations."""
        with contextlib.suppress(Exception):
            if self._client:
                self._client.stop_channels()
        with contextlib.suppress(Exception):
            if self._km:
                self._km.shutdown_kernel(now=True)
        with contextlib.suppress(Exception):
            if self._km:
                self._km.cleanup_resources()
        # Start fresh
        self._start_kernel()

    def close(self):

        with contextlib.suppress(Exception):
            if self._client:
                self._client.stop_channels()

        if self._owns_kernel and self._km is not None:
            with contextlib.suppress(Exception):
                self._km.shutdown_kernel(now=True)

            with contextlib.suppress(Exception):
                self._km.cleanup_resources()

    def reset(self):
        
        self.execute(
            '%reset -f\n'
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import functools\n'
            'import fractions\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
        )

    def __del__(self):

        self.close()

In [ ]:
class AIMO3Tool:

    def __init__(self, local_jupyter_timeout: float, tool_prompt: str, sandbox=None):

        self._local_jupyter_timeout = local_jupyter_timeout
        self._tool_prompt = tool_prompt
        self._jupyter_session = sandbox
        
        self._owns_session = sandbox is None
        
        self._execution_lock = threading.Lock()
        self._init_lock = threading.Lock()

    def _ensure_session(self):

        if self._jupyter_session is None:
            with self._init_lock:
                if self._jupyter_session is None:
                    self._jupyter_session = AIMO3Sandbox(timeout=self._local_jupyter_timeout)

    def _ensure_last_print(self, code: str) -> str:

        lines = code.strip().split('\n')

        if not lines:
            return code

        last_line = lines[-1].strip()

        if 'print' in last_line or 'import' in last_line:
            return code

        if not last_line:
            return code

        if last_line.startswith('#'):
            return code

        # Skip wrapping assignments (x = 5 → print(x = 5) causes TypeError)
        # Also skip augmented assignments (+=, -=, etc.), for/while/if/else/with/try/class/def, and decorators
        if (re.search(r'(?<![=!<>])=(?!=)', last_line)
                or last_line.split()[0] in ('for', 'while', 'if', 'elif', 'else:', 'with', 'try:', 'except', 'except:', 'finally:', 'class', 'def', 'return', 'raise', 'del', 'assert', 'pass', 'break', 'continue')
                or last_line.startswith('@')):
            return code

        lines[-1] = 'print(' + last_line + ')'

        return '\n'.join(lines)

    @property
    def instruction(self) -> str:

        return self._tool_prompt

    @property
    def tool_config(self) -> ToolNamespaceConfig:

        return ToolNamespaceConfig(
            name='python', 
            description=self.instruction, 
            tools=[]
        )

    def _make_response(self, output: str, channel: str | None = None) -> Message:

        content = TextContent(text=output)
        author = Author(role=Role.TOOL, name='python')
        message = Message(author=author, content=[content]).with_recipient('assistant')

        if channel:
            message = message.with_channel(channel)

        return message

    def process_sync_plus(self, message: Message) -> list[Message]:

        self._ensure_session()
        raw_script = message.content[0].text
        # Strip markdown code fences if model wraps code in them
        stripped = raw_script.strip()
        if stripped.startswith('```'):
            lines = raw_script.split('\n')
            if lines[0].strip().startswith('```'):
                lines = lines[1:]
            if lines and lines[-1].strip() == '```':
                lines = lines[:-1]
            raw_script = '\n'.join(lines)
        final_script = self._ensure_last_print(raw_script)

        with self._execution_lock:
            try:
                output = self._jupyter_session.execute(final_script)

            except TimeoutError as exc:
                output = f'[ERROR] {exc}'

        # Cap output length to prevent notebook bloat (HiGHS MILP can dump 10K+ lines)
        max_output_chars = 8000
        if len(output) > max_output_chars:
            truncated = len(output) - max_output_chars
            output = output[:max_output_chars // 2] + f'\n... [{truncated} chars truncated] ...\n' + output[-max_output_chars // 2:]

        return [self._make_response(output, channel=message.channel)]

In [ ]:
class AIMO3Solver:

    def __init__(self, cfg, port: int = 8000):
    
        self.cfg = cfg
        self.port = port
        self.base_url = f'http://0.0.0.0:{port}/v1'
        self.api_key = 'sk-local'
        self.template = AIMO3Template()
        self.encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
        self.stop_token_ids = self.encoding.stop_tokens_for_assistant_actions()
    
        self._preload_model_weights()
        
        self.server_process = self._start_server()
    
        self.client = OpenAI(
            base_url=self.base_url, 
            api_key=self.api_key, 
            timeout=self.cfg.session_timeout
        )
    
        self._wait_for_server()
        self._initialize_kernels()
    
        self.notebook_start_time = time.time()
        self.problems_remaining = 50
    
    def _preload_model_weights(self) -> None:
    
        print(f'Loading model weights from {self.cfg.model_path} into OS Page Cache...')
        start_time = time.time()
        
        files_to_load = []
        total_size = 0
    
        for root, _, files in os.walk(self.cfg.model_path):
            for file_name in files:
                file_path = os.path.join(root, file_name)
    
                if os.path.isfile(file_path):
                    files_to_load.append(file_path)
                    total_size += os.path.getsize(file_path)
    
        def _read_file(path: str) -> None:
    
            with open(path, 'rb') as file_object:
                while file_object.read(1024 * 1024 * 1024):
                    pass
    
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            list(executor.map(_read_file, files_to_load))
    
        elapsed = time.time() - start_time
        print(f'Processed {len(files_to_load)} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')
    
    def _start_server(self) -> subprocess.Popen:
    
        cmd = [
            sys.executable, 
            '-m', 
            'vllm.entrypoints.openai.api_server', 
            '--seed', 
            str(self.cfg.seed), 
            '--model', 
            self.cfg.model_path, 
            '--served-model-name', 
            self.cfg.served_model_name, 
            '--tensor-parallel-size', 
            '1', 
            '--max-num-seqs', 
            str(self.cfg.batch_size), 
            '--gpu-memory-utilization', 
            str(self.cfg.gpu_memory_utilization), 
            '--host', 
            '0.0.0.0', 
            '--port', 
            str(self.port), 
            '--dtype', 
            self.cfg.dtype, 
            '--kv-cache-dtype', 
            self.cfg.kv_cache_dtype, 
            '--max-model-len', 
            str(self.cfg.context_tokens), 
            '--stream-interval', 
            str(self.cfg.stream_interval), 
            '--async-scheduling', 
            '--disable-log-stats', 
            '--enable-prefix-caching'
        ]
    
        self.log_file = open('vllm_server.log', 'w')
    
        return subprocess.Popen(
            cmd, 
            stdout=self.log_file, 
            stderr=subprocess.STDOUT, 
            start_new_session=True
        )
    
    def _wait_for_server(self):
    
        print('Waiting for vLLM server...')
        start_time = time.time()
    
        for _ in range(self.cfg.server_timeout):
            return_code = self.server_process.poll()
    
            if return_code is not None:
                self.log_file.flush()
    
                with open('vllm_server.log', 'r') as log_file:
                    logs = log_file.read()
    
                raise RuntimeError(f'Server died with code {return_code}. Full logs:\n{logs}\n')
    
            try:
                self.client.models.list()
                elapsed = time.time() - start_time
                print(f'Server is ready (took {elapsed:.2f} seconds).\n')
    
                return
    
            except Exception:
                time.sleep(1)
    
        raise RuntimeError('Server failed to start (timeout).\n')
    
    def _initialize_kernels(self) -> None:
    
        print(f'Initializing {self.cfg.workers} persistent Jupyter kernels...')
        start_time = time.time()
    
        self.sandbox_pool = queue.Queue()
    
        def _create_sandbox():
            
            return AIMO3Sandbox(timeout=self.cfg.jupyter_timeout)
    
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            futures = [executor.submit(_create_sandbox) for _ in range(self.cfg.workers)]
    
            for future in as_completed(futures):
                self.sandbox_pool.put(future.result())
    
        elapsed = time.time() - start_time
        print(f'Kernels initialized in {elapsed:.2f} seconds.\n')
    
    def _scan_for_answer(self, text: str) -> tuple[int | None, float]:
        """Extract answer from text. Returns (answer, confidence).
        confidence=1.0 for \\boxed{}, 0.7 for \\Vboxed{} or fallback patterns."""

        def _try_extract(val_str: str) -> int | None:
            try:
                clean = val_str.replace(',', '').strip()
                val = int(clean)
                if 0 <= val <= 99999:
                    return val
            except ValueError:
                pass
            return None

        # Priority 1: \boxed{N} — full confidence (last match wins)
        matches = re.findall(r'\\boxed\s*\{\s*([0-9,]+)\s*\}', text)
        if matches:
            val = _try_extract(matches[-1])
            if val is not None:
                return val, 1.0

        # Priority 2: \Vboxed{N} — checkpoint, reduced confidence (last match wins)
        matches = re.findall(r'\\Vboxed\s*\{\s*([0-9,]+)\s*\}', text)
        if matches:
            val = _try_extract(matches[-1])
            if val is not None:
                return val, 0.7

        # Priority 3: 'final answer is X'
        matches = re.findall(r'final\s+answer\s+is\s*([0-9,]+)', text, re.IGNORECASE)
        if matches:
            val = _try_extract(matches[-1])
            if val is not None:
                return val, 0.7

        # Priority 4: 'the answer is X' / 'answer is X'
        matches = re.findall(r'(?:the\s+)?answer\s+is\s*:?\s*\*?\*?\s*([0-9,]+)', text, re.IGNORECASE)
        if matches:
            val = _try_extract(matches[-1])
            if val is not None:
                return val, 0.7

        # Priority 5: 'answer: X' / 'Answer = X'
        matches = re.findall(r'answer\s*[:=]\s*\*?\*?\s*([0-9,]+)', text, re.IGNORECASE)
        if matches:
            val = _try_extract(matches[-1])
            if val is not None:
                return val, 0.7

        # Priority 6: Any numeric content inside \boxed{} (handles LaTeX formatting)
        matches = re.findall(r'\\boxed\s*\{([^}]+)\}', text)
        if matches:
            nums = re.findall(r'(\d+)', matches[-1])
            if nums:
                val = _try_extract(nums[-1])
                if val is not None:
                    return val, 0.5

        return None, 0.0

    
    def _compute_mean_entropy(self, logprobs_buffer: list) -> float:
    
        if not logprobs_buffer:
            return float('inf')
    
        total_entropy = 0.0
        token_count = 0
    
        for top_logprobs_dict in logprobs_buffer:
            
            if not isinstance(top_logprobs_dict, dict):
                continue
            
            if not top_logprobs_dict:
                continue
            
            token_entropy = 0.0
            
            for token_str, log_prob in top_logprobs_dict.items():
                prob = math.exp(log_prob)
                
                if prob > 0:
                    token_entropy -= prob * math.log2(prob)
            
            total_entropy += token_entropy
            token_count += 1
    
        if token_count == 0:
            return float('inf')
    
        return total_entropy / token_count
    
    def _process_attempt(
        self, 
        problem: str, 
        system_prompt: str, 
        attempt_index: int, 
        stop_event: threading.Event, 
        deadline: float,
        temperature: float = None
    ) -> dict:
    
        if stop_event.is_set() or time.time() > deadline:
            return {
                'Attempt': attempt_index + 1, 
                'Answer': None, 
                'Python Calls': 0, 
                'Python Errors': 0, 
                'Response Length': 0, 
                'Entropy': float('inf'),
                'Temperature': temperature,
                'Confidence': 0.0,
                'Source': 'skipped',
                'Stop_Reason': 'early_skip',
                'Successful_Code': 0,
                'Conversation': [],
                'Time': 0.0
            }
    
        local_tool = None
        sandbox = None
        python_calls = 0
        python_errors = 0
        total_tokens = 0
        final_answer = None
        answer_confidence = 0.0
        answer_source = 'none'
        
        logprobs_buffer = []
        conversation_log = []
        stop_reason = 'max_turns'
        turn_num = 0
        consecutive_errors = 0
        attempt_start = time.time()
    
        attempt_seed = (self.cfg.seed + attempt_index) ** 2
    
        try:
            sandbox = self.sandbox_pool.get(timeout=self.cfg.sandbox_timeout)
    
            local_tool = AIMO3Tool(
                local_jupyter_timeout=self.cfg.jupyter_timeout, 
                tool_prompt=self.cfg.tool_prompt, 
                sandbox=sandbox
            )

            # Inject deadline into sandbox so model can check time remaining
            sandbox.execute(f'import time as _time; _DEADLINE = {deadline}')
    
            encoding = self.encoding
            messages = self.template.apply_chat_template(
                system_prompt, 
                problem, 
                local_tool.tool_config
            )
    
            conversation = Conversation.from_messages(messages)
    
            for _ in range(self.cfg.turns):
                turn_num += 1
                time_left = deadline - time.time()
                if stop_event.is_set() or time_left <= 0:
                    break

                # Hard takeover at ≤5s: scan conversation for any answer, or force emergency completion
                if time_left <= 5 and final_answer is None:
                    # Try to salvage an answer from existing conversation
                    if conversation_log:
                        full_text = ' '.join(
                            entry.get('text', '') + ' ' + entry.get('code', '') + ' ' + entry.get('output', '')
                            for entry in conversation_log
                        )
                        cand, conf = self._scan_for_answer(full_text)
                        if cand is not None:
                            final_answer, answer_confidence = cand, conf
                            answer_source = 'emergency_checkpoint'
                            stop_reason = 'emergency_deadline'
                            break

                    # Emergency: tiny completion to force a \Vboxed{N}
                    flush_msgs = self.template.apply_chat_template(
                        system_prompt,
                        'You are out of time. Output ONLY \\Vboxed{N} where N is your best integer answer (0-99999). No other text.',
                        local_tool.tool_config
                    )
                    flush_conv = Conversation.from_messages(flush_msgs)
                    flush_ids = encoding.render_conversation_for_completion(flush_conv, Role.ASSISTANT)
                    flush = self.client.completions.create(
                        model=self.cfg.served_model_name,
                        temperature=0.0,
                        max_tokens=64,
                        prompt=flush_ids,
                        seed=attempt_seed,
                        stream=False,
                        extra_body={'min_p': self.cfg.min_p, 'stop_token_ids': self.stop_token_ids, 'return_token_ids': True}
                    )
                    flush_text = flush.choices[0].text or ''
                    cand, conf = self._scan_for_answer(flush_text)
                    if cand is not None:
                        final_answer, answer_confidence = cand, conf
                        answer_source = 'emergency_forced'
                        stop_reason = 'emergency_deadline'
                    break

                prompt_ids = encoding.render_conversation_for_completion(conversation, Role.ASSISTANT)
                max_tokens = self.cfg.context_tokens - len(prompt_ids)
    
                if max_tokens < self.cfg.buffer_tokens:
                    break
    
                stream = self.client.completions.create(
                    model=self.cfg.served_model_name, 
                    temperature=(temperature if temperature is not None else self.cfg.temperature), 
                    logprobs=self.cfg.top_logprobs, 
                    max_tokens=max_tokens, 
                    prompt=prompt_ids, 
                    seed=attempt_seed, 
                    stream=True, 
                    extra_body={
                        'min_p': self.cfg.min_p, 
                        'stop_token_ids': self.stop_token_ids, 
                        'return_token_ids': True
                    }
                )
    
                try:
                    token_buffer = []
                    text_chunks = []
    
                    for chunk in stream:
                        if stop_event.is_set() or time.time() > deadline:
                            break
    
                        new_tokens = chunk.choices[0].token_ids
                        new_text = chunk.choices[0].text
    
                        if new_tokens:
                            token_buffer.extend(new_tokens)
                            total_tokens += len(new_tokens)
                            text_chunks.append(new_text)
                            
                            chunk_logprobs = chunk.choices[0].logprobs
                            
                            if chunk_logprobs is not None:
                                if chunk_logprobs.top_logprobs:
                                    logprobs_buffer.extend(chunk_logprobs.top_logprobs)
    
                        if '}' in new_text:
                            search_text = ''.join(text_chunks[-self.cfg.search_tokens:])
                            answer, conf = self._scan_for_answer(search_text)
    
                            if answer is not None:
                                final_answer = answer
                                answer_confidence = conf
                                answer_source = 'streaming'
                                break
    
                finally:
                    stream.close()
    
                if final_answer is not None:
                    break
    
                if not token_buffer:
                    break
    
                new_messages = encoding.parse_messages_from_completion_tokens(token_buffer, Role.ASSISTANT)
                conversation.messages.extend(new_messages)
                last_message = new_messages[-1]

                # Log reasoning text
                msg_text = last_message.content[0].text if last_message.content else ''
                conversation_log.append({
                    'turn': turn_num,
                    'type': 'reasoning' if last_message.recipient != 'python' else 'code_call',
                    'text': msg_text if last_message.recipient != 'python' else '',
                    'code': msg_text if last_message.recipient == 'python' else '',
                    'output': '',
                    'error': False
                })
    
                if last_message.channel == 'final':
                    answer_text = last_message.content[0].text
                    final_answer, answer_confidence = self._scan_for_answer(answer_text)
                    answer_source = 'final_channel'
                    break
    
                if last_message.recipient == 'python':
                    python_calls += 1
                    raw_code = last_message.content[0].text

                    # Block code execution in last 5s — force checkpoint instead
                    time_left = deadline - time.time()
                    if time_left <= 5:
                        tool_responses = [local_tool._make_response(
                            f'[DEADLINE] {time_left:.0f}s left. Code execution blocked. Write \\Vboxed{{N}} with your best answer NOW.',
                            channel=last_message.channel
                        )]
                    else:
                        tool_responses = local_tool.process_sync_plus(last_message)

                    response_text = tool_responses[0].content[0].text

                    conversation_log.append({
                        'turn': turn_num,
                        'type': 'code_call',
                        'code': raw_code,
                        'output': response_text,
                        'error': response_text.startswith('[ERROR]') or 'Traceback' in response_text or 'Error:' in response_text
                    })
    
                    is_error = response_text.startswith('[ERROR]') or 'Traceback' in response_text or 'Error:' in response_text
                    if is_error:
                        python_errors += 1
                        consecutive_errors += 1
                        if consecutive_errors >= 8:
                            stop_reason = 'error_cascade'
                            break
                    else:
                        consecutive_errors = 0

                    # Exam bell: 3-tier time warnings in tool responses
                    # Tier 1 (<100s): heads up, start wrapping up
                    # Tier 2 (<30s): must Vboxed now, keep working but checkpoint first
                    # Tier 3 (<5s): hard takeover (handled at top of turn loop)
                    time_remaining = deadline - time.time()
                    if time_remaining < 30:
                        bell = '\n\n[URGENT — <30s remaining. You MUST write \\Vboxed{YOUR_ANSWER} with your best answer RIGHT NOW before continuing.]'
                        tool_responses = [local_tool._make_response(response_text + bell, channel=last_message.channel)]
                    elif time_remaining < 100:
                        bell = f'\n\n[NOTICE — ~{time_remaining:.0f}s remaining on this problem.]'
                        tool_responses = [local_tool._make_response(response_text + bell, channel=last_message.channel)]

                    conversation.messages.extend(tool_responses)
    
        except Exception as exc:
            python_errors += 1
    
        finally:
            if sandbox is not None:
                try:
                    sandbox.reset()
                    self.sandbox_pool.put(sandbox)
                except Exception:
                    try:
                        sandbox.close()
                    except Exception:
                        pass
                    try:
                        self.sandbox_pool.put(AIMO3Sandbox(timeout=self.cfg.jupyter_timeout))
                    except Exception:
                        pass
    
        mean_entropy = self._compute_mean_entropy(logprobs_buffer)
    
        attempt_elapsed = time.time() - attempt_start
    
        # If no answer found via streaming, scan full conversation for Vboxed checkpoints
        if final_answer is None and conversation_log:
            full_text = ' '.join(entry.get('text', '') + ' ' + entry.get('code', '') + ' ' + entry.get('output', '') for entry in conversation_log)
            final_answer, answer_confidence = self._scan_for_answer(full_text)
            answer_source = 'fallback'

        return {
            'Attempt': attempt_index + 1, 
            'Response Length': total_tokens, 
            'Python Calls': python_calls, 
            'Python Errors': python_errors, 
            'Entropy': mean_entropy, 
            'Temperature': temperature,
            'Answer': final_answer,
            'Confidence': answer_confidence,
            'Source': answer_source,
            'Stop_Reason': stop_reason,
            'Successful_Code': python_calls - python_errors,
            'Conversation': conversation_log,
            'Time': round(attempt_elapsed, 1)
        }
    
    def _select_answer(self, detailed_results: list) -> int:

        answer_weights = defaultdict(float)
        answer_votes = defaultdict(float)

        for result in detailed_results:
            answer = result['Answer']
            entropy = result['Entropy']
            confidence = result.get('Confidence', 1.0)
            successful_code = result.get('Successful_Code', 0)

            if answer is not None:
                # Base weight: inverse entropy
                if not math.isfinite(entropy) or entropy <= 0:
                    entropy = float('inf')
                weight = 1.0 / max(entropy, 1e-9)


                # Confidence is applied via vote count (line 526), not weight

                # Downweight attempts with zero successful code executions
                if successful_code == 0:
                    weight *= 0.5

                answer_weights[answer] += weight
                answer_votes[answer] += confidence

        scored_answers = []

        for answer, total_weight in answer_weights.items():
            scored_answers.append({
                'answer': answer,
                'votes': answer_votes[answer],
                'score': total_weight
            })

        # Deterministic tie-breaking: votes (desc), then score (desc), then answer value (desc)
        scored_answers.sort(key=lambda x: (x['votes'], x['score'], x['answer']), reverse=True)

        vote_data = []

        for item in scored_answers:
            vote_data.append((
                item['answer'],
                item['votes'],
                item['score']
            ))

        vote_dataframe = pd.DataFrame(
            vote_data,
            columns=['Answer', 'Votes', 'Score']
        )

        vote_dataframe = vote_dataframe.round({'Score': 3})
        display(vote_dataframe)

        if not scored_answers:
            print('\nFinal Answer: 0\n')
            return 0

        final_answer = scored_answers[0]['answer']    
        print(f'\nFinal Answer: {final_answer}\n')

        return final_answer
    
    def solve_problem(self, problem: str) -> int:
    
        print(f'\nProblem: {problem}\n')
        
        user_input = f'{problem} {self.cfg.preference_prompt}'
    
        elapsed_global = time.time() - self.notebook_start_time
        time_left = self.cfg.notebook_limit - elapsed_global

        budget = time_left / max(1, self.problems_remaining)
        budget = min(budget, self.cfg.problem_timeout)
        budget = max(budget, 60)
    
        deadline = time.time() + budget
    
        print(f'Budget: {budget:.2f} seconds | Deadline: {deadline:.2f}\n')
    
        tasks = []
    
        for attempt_index in range(self.cfg.attempts):
            temp = self.cfg.temp_schedule[attempt_index] if hasattr(self.cfg, 'temp_schedule') and self.cfg.temp_schedule else self.cfg.temperature
            tasks.append((self.cfg.system_prompt, attempt_index, temp))
    
        detailed_results = []
        valid_answers = []
    
        stop_event = threading.Event()
    
        executor = ThreadPoolExecutor(max_workers=self.cfg.workers)
    
        try:
            futures = []
    
            for (system_prompt, attempt_index, temp) in tasks:
                future = executor.submit(
                    self._process_attempt, 
                    user_input, 
                    system_prompt, 
                    attempt_index, 
                    stop_event, 
                    deadline,
                    temp
                )
    
                futures.append(future)
    
            for future in as_completed(futures):
                try:
                    result = future.result()
                    detailed_results.append(result)
    
                    if result['Answer'] is not None:
                        valid_answers.append(result['Answer'])
    
                except Exception as exc:
                    print(f'Future failed: {exc}')
                    continue
    
        finally:
            stop_event.set()
            executor.shutdown(wait=True, cancel_futures=True)
            
            self.problems_remaining = max(0, self.problems_remaining - 1)
    
        if detailed_results:
            results_dataframe = pd.DataFrame(detailed_results)
            results_dataframe['Entropy'] = results_dataframe['Entropy'].round(3)
            results_dataframe['Answer'] = results_dataframe['Answer'].astype('Int64')
            
            display(results_dataframe)
    
        if not valid_answers:
            print('\nResult: 0\n')
    
            return 0
    
        return self._select_answer(detailed_results)
    
    def __del__(self):
    
        if hasattr(self, 'server_process'):
            self.server_process.terminate()
            self.server_process.wait()
    
        if hasattr(self, 'log_file'):
            self.log_file.close()
    
        if hasattr(self, 'sandbox_pool'):
            while not self.sandbox_pool.empty():
                try:
                    sb = self.sandbox_pool.get_nowait()
                    sb.close()
    
                except Exception:
                    pass


In [ ]:
solver = AIMO3Solver(CFG)

In [ ]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    
    id_value = id_.item()
    question_text = question.item()
    
    gc.disable()
    
    final_answer = solver.solve_problem(question_text)
    
    gc.enable()
    gc.collect()
    
    return pl.DataFrame({'id': id_value, 'answer': final_answer})

In [ ]:
inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()

else:
    # Find test.csv — path varies between test runs and competition reruns
    test_csv_candidates = [
        '/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv',
        '/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/test.csv',
    ]
    test_csv = next((p for p in test_csv_candidates if os.path.isfile(p)), test_csv_candidates[0])
    print(f'Using test CSV: {test_csv}')

    inference_server.run_local_gateway(
        (test_csv,)
    )

In [ ]:
import gc
# =============================================================================
# ENHANCED TEST FRAMEWORK — Maximum Diagnostic Logging
# =============================================================================
# Diagnostic output is written to /kaggle/working/diagnostic.log (persists as kernel output)
# This avoids Kaggle's stdout truncation and UI hanging on large logs.

import sys

class TeeLogger:
    """Write to log file only. Avoids 80MB notebook from stdout duplication."""
    def __init__(self, filepath='/kaggle/working/diagnostic.log'):
        self.terminal = sys.stdout
        self.log = open(filepath, 'w', buffering=1)  # line-buffered for live output
    def write(self, message):
        self.log.write(message)
    def flush(self):
        self.log.flush()

sys.stdout = TeeLogger()
sys.stderr = TeeLogger('/kaggle/working/diagnostic_stderr.log')

# =============================================================================
# v24 Test Framework
#   Tier 0: Sanity check — 5 diverse problems, single run (pre-submission check)
#   Tier 1: Remaining previous failures + dodgy wins, single run
#   Tier 2: Full val bench remaining (loaded from CSV)
#
# TEST_TIER controls which tiers run:
#   0 = Tier 0 only (5 problems) — fast sanity check
#   1 = Tier 0 + 1 (all previous failures) — regression
#   2 = Tier 0 + 1 + 2 (full val bench)
# =============================================================================

TEST_TIER = 2  # v29: full tier run with all fixes

if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):

    def _print_attempt_full(r, expected):
        """Print complete diagnostic for one attempt — every step, no truncation."""
        attempt = r.get('Attempt', '?')
        answer = r.get('Answer', None)
        entropy = r.get('Entropy', float('inf'))
        py_calls = r.get('Python Calls', 0)
        py_errors = r.get('Python Errors', 0)
        resp_len = r.get('Response Length', 0)
        temp = r.get('Temperature', '?')
        elapsed = r.get('Time', '?')
        status = 'CORRECT' if answer == expected else ('WRONG' if answer is not None else 'NONE')
        print(f'\n    --- Attempt {attempt} [{status}] answer={answer} expected={expected} temp={temp} entropy={entropy:.3f} ---')
        print(f'        python_calls={py_calls} errors={py_errors} tokens={resp_len} time={elapsed}s')

        conv = r.get('Conversation', [])
        if conv:
            imports_found = set()
            for entry in conv:
                if entry.get('type') == 'code_call':
                    code = entry.get('code', '')
                    for line in code.split('\n'):
                        line = line.strip()
                        if line.startswith('import ') or line.startswith('from '):
                            imports_found.add(line)
            if imports_found:
                print(f'        Libraries: {", ".join(sorted(imports_found))}')

            for entry in conv:
                turn = entry.get('turn', '?')
                etype = entry.get('type', '?')
                if etype == 'reasoning':
                    text = entry.get('text', '')
                    print(f'\n        [Turn {turn} REASONING] ({len(text)} chars)')
                    print(f'        {text}')
                elif etype == 'code_call':
                    code = entry.get('code', '')
                    output = entry.get('output', '')
                    is_error = entry.get('error', False)
                    err_marker = ' [ERROR]' if is_error else ''
                    print(f'\n        [Turn {turn} CODE{err_marker}]')
                    print(f'        >>> Code:')
                    for cline in code.split('\n'):
                        print(f'            {cline}')
                    print(f'        >>> Output:')
                    for oline in output.split('\n')[:50]:
                        print(f'            {oline}')
                    if len(output.split(chr(10))) > 50:
                        print(f'            ... ({len(output.split(chr(10)))} lines total)')

    def _get_gpu_stats():
        """Get GPU memory stats if available."""
        try:
            import torch
            if torch.cuda.is_available():
                alloc = torch.cuda.memory_allocated() / 1e9
                reserved = torch.cuda.memory_reserved() / 1e9
                return f'GPU mem: {alloc:.1f}GB alloc / {reserved:.1f}GB reserved'
        except Exception:
            pass
        return 'GPU stats unavailable'

    def run_test_batch(name, problems, solver):
        """Run a batch of test problems with full diagnostics."""
        print(f'\n{"="*80}')
        print(f'  BATCH: {name} ({len(problems)} problems)')
        print(f'  {_get_gpu_stats()}')
        print(f'{"="*80}')

        batch_correct = 0
        batch_total = len(problems)
        batch_results = []
        batch_start = time.time()

        for idx, p in enumerate(problems):
            pid = p['id']
            expected = p['answer']
            problem_text = p['problem']

            print(f'\n{"─"*70}')
            print(f'  [{idx+1}/{batch_total}] Problem {pid} | Expected: {expected}')
            print(f'{"─"*70}')

            prob_start = time.time()
            predicted = solver.solve_problem(problem_text)
            prob_elapsed = time.time() - prob_start

            correct = (predicted == expected)
            batch_correct += int(correct)
            status = 'CORRECT' if correct else 'WRONG'

            result_entry = {
                'id': pid,
                'predicted': predicted,
                'expected': expected,
                'correct': correct,
                'time': round(prob_elapsed, 1),
            }

            detailed = getattr(solver, '_last_detailed_results', [])
            if detailed:
                answers = [r['Answer'] for r in detailed if r.get('Answer') is not None]
                from collections import Counter
                vote_dist = dict(Counter(answers).most_common())
                result_entry['votes'] = vote_dist
                result_entry['num_answers'] = len(answers)
                result_entry['num_none'] = sum(1 for r in detailed if r.get('Answer') is None)
                result_entry['num_errors'] = sum(r.get('Python Errors', 0) for r in detailed)

                print(f'\n  STATUS: {status} | Predicted: {predicted} | Expected: {expected} | Time: {prob_elapsed:.1f}s')
                print(f'  Votes: {vote_dist} | Answers: {len(answers)} | Nones: {result_entry["num_none"]} | Errors: {result_entry["num_errors"]}')

                if not correct:
                    print(f'\n  WRONG ANSWER ANALYSIS:')
                    for r in detailed:
                        if r.get('Answer') is not None and r['Answer'] != expected:
                            print(f'    Attempt {r["Attempt"]}: answered {r["Answer"]} (entropy={r["Entropy"]:.3f})')
                    correct_attempts = [r for r in detailed if r.get('Answer') == expected]
                    if correct_attempts:
                        print(f'    {len(correct_attempts)} attempt(s) got the right answer but were outvoted')

                for r in detailed:
                    _print_attempt_full(r, expected)
            else:
                print(f'\n  STATUS: {status} | Predicted: {predicted} | Expected: {expected} | Time: {prob_elapsed:.1f}s')

            batch_results.append(result_entry)
            gc.collect()

            running_pct = batch_correct / (idx + 1) * 100
            print(f'\n  Running: {batch_correct}/{idx+1} ({running_pct:.0f}%) | Elapsed: {time.time()-batch_start:.0f}s')

        batch_elapsed = time.time() - batch_start
        print(f'\n{"="*80}')
        print(f'  BATCH COMPLETE: {name}')
        print(f'  Score: {batch_correct}/{batch_total} ({batch_correct/max(batch_total,1)*100:.0f}%)')
        print(f'  Time: {batch_elapsed:.0f}s')
        if batch_results:
            times = [r['time'] for r in batch_results]
            print(f'  Timing: min={min(times):.0f}s  max={max(times):.0f}s  median={sorted(times)[len(times)//2]:.0f}s')
        print(f'{"="*80}')

        return batch_correct, batch_total, batch_results

    def load_csv_problems(problems_csv, answers_csv):
        """Load problems and answers from CSV files."""
        import csv
        problems = {}
        answers = {}
        with open(problems_csv, 'r') as f:
            reader = csv.DictReader(f)
            for row in reader:
                problems[row['id']] = row['problem']
        with open(answers_csv, 'r') as f:
            reader = csv.DictReader(f)
            for row in reader:
                answers[row['id']] = int(row['answer'])
        return [{'id': pid, 'problem': problems[pid], 'answer': answers[pid]}
                for pid in problems if pid in answers]

    def load_csv_problems_by_ids(problems_csv, answers_csv, target_ids):
        """Load specific problems by ID from CSV files."""
        import csv
        problems = {}
        answers = {}
        with open(problems_csv, 'r') as f:
            reader = csv.DictReader(f)
            for row in reader:
                if row['id'] in target_ids:
                    problems[row['id']] = row['problem']
        with open(answers_csv, 'r') as f:
            reader = csv.DictReader(f)
            for row in reader:
                if row['id'] in target_ids:
                    answers[row['id']] = int(row['answer'])
        return [{'id': pid, 'problem': problems[pid], 'answer': answers[pid]}
                for pid in target_ids if pid in problems and pid in answers]

    # =========================================================================
    #  TIER 0: Sanity Check — 2 toughest v23 failures
    #  Validates code execution, voting, extraction under stress.
    # =========================================================================

    TIER_0_HARDCODED = []

    TIER_0_CSV_IDS = [
        'dbbfe8',   # v31: lost by 1 vote (8×7 vs 22×6), exp=22
        '3b88b3',   # v31: unanimous wrong 982×16, exp=979 (off by 3)
    ]

    # =========================================================================
    #  TIER 0.5: Historical hard problems — double run (2x for consistency)
    #  Problems that are known capability limits. Run twice to check variance.
    # =========================================================================

    TIER_05_HARDCODED = [
        # Norwegian numbers — 11 unique answers in v31. Capability limit.
        {'id': '86e8e5', 'problem': 'Let $n \\geq 6$ be a positive integer. We call a positive integer $n$-Norwegian if it has three distinct positive divisors whose sum is equal to $n$. Let $f(n)$ denote the smallest $n$-Norwegian positive integer. Let $M=3^{2025!}$ and for a non-negative integer $c$ define $g(c)=\\frac{1}{2025!}\\left\\lfloor \\frac{2025! f(M+c)}{M}\\right\\rfloor.$ We can write $g(0)+g(4M)+g(1848374)+g(10162574)+g(265710644)+g(44636594)=\\frac{p}{q}$ where $p$ and $q$ are coprime positive integers. What is the remainder when $p+q$ is divided by $99991$?', 'answer': 8687},
    ]

    # =========================================================================
    #  TIER 1: Remaining previous failures + dodgy wins (single run)
    # =========================================================================

    # IDs of problems wrong in v31 (updated from v23)
    V31_WRONG_IDS = [
        '1ec970', '23586c', '26bee3', '29714f', '3980cd', '3b88b3',
        '414a5b', '673b29', '9010d9', 'a824c1',
        'a9dbc8', 'ae2add', 'aff75c', 'dbbfe8'
    ]

    # Tier 1.5: Was wrong in v23, now correct in v31 (monitor for regressions)
    TIER_15_IDS = [
        '21fb4e',   # v23=17→v31=16 (correct). Was toughest problem.
        '89c921',   # v23 wrong→v31 correct.
    ]

    # Dodgy wins + regressions
    DODGY_WIN_IDS = [
        '32690e',   # 6:4 margin, off-by-one risk (4050 vs 4051)
    ]

    # =========================================================================
    #  RUN TIERS
    # =========================================================================
    all_results = []
    total_correct = 0
    total_problems = 0
    suite_start = time.time()

    # --- TIER 0: Sanity check — 5 diverse problems, single run ---
    problems_csv = '/kaggle/input/aimo3-test-data/test_problems.csv'
    answers_csv = '/kaggle/input/aimo3-test-data/test_answers.csv'

    tier0_csv = []
    if TIER_0_CSV_IDS:
        if os.path.isfile(problems_csv) and os.path.isfile(answers_csv):
            tier0_csv = load_csv_problems_by_ids(problems_csv, answers_csv, set(TIER_0_CSV_IDS))
        else:
            print(f'WARNING: CSV not found at {problems_csv}, tier 0 CSV problems skipped')

    tier0_problems = TIER_0_HARDCODED + tier0_csv
    print(f'\n=== PROBLEM MANIFEST ===')
    print(f'  Tier 0: {[p["id"] for p in tier0_problems]}')
    if TEST_TIER >= 0.5:
        print(f'  Tier 0.5: {[p["id"] for p in TIER_05_HARDCODED]} x2')
    print(f'  Tier 1: {len(V31_WRONG_IDS)} wrong + {len(TIER_15_IDS)} monitor + {len(DODGY_WIN_IDS)} dodgy (minus tier 0/0.5 overlap)')
    print(f'  Tier 2: up to 35 from val bench')
    print(f'  TEST_TIER={TEST_TIER}')
    print(f'=== END MANIFEST ===')

    # Count total problems across all tiers for budget calculation
    total_test_problems = len(tier0_problems)
    if TEST_TIER >= 0.5:
        total_test_problems += len(TIER_05_HARDCODED) * 2
    if TEST_TIER >= 1:
        total_test_problems += len(tier1_ids) if 'tier1_ids' in dir() else 14
    if TEST_TIER >= 2:
        total_test_problems += 35  # MAX_TIER2 cap
    solver.problems_remaining = total_test_problems
    print(f'\nTotal test problems: ~{total_test_problems} (problems_remaining set)')

    if TEST_TIER >= 0:
        print(f'\nTier 0: {len(tier0_problems)} problems (sanity check)')
        c0, t0, r0 = run_test_batch('TIER 0 — SANITY CHECK', tier0_problems, solver)
        total_correct += c0
        total_problems += t0
        all_results.extend(r0)
    else:
        print(f'\nSKIPPING all tiers (TEST_TIER={TEST_TIER})')

    # --- TIER 0.5: Historical hard problems, double run ---
    if TEST_TIER >= 0.5:
        tier05_problems = TIER_05_HARDCODED
        # Run twice for variance check
        tier05_double = tier05_problems + tier05_problems
        print(f'\nTier 0.5: {len(tier05_problems)} problems x2 runs ({len(tier05_double)} total)')
        c05, t05, r05 = run_test_batch('TIER 0.5 — HISTORICAL HARD (2x)', tier05_double, solver)
        total_correct += c05
        total_problems += t05
        all_results.extend(r05)
    else:
        print(f'\nSKIPPING Tier 0.5 (TEST_TIER={TEST_TIER})')

    # --- TIER 1: Remaining previous failures + dodgy wins, single run ---
    if TEST_TIER >= 1:
        tier0_ids = set(p['id'] for p in tier0_problems) | set(p['id'] for p in TIER_05_HARDCODED)
        csv_ids = (set(V31_WRONG_IDS) | set(DODGY_WIN_IDS) | set(TIER_15_IDS)) - tier0_ids
        if os.path.isfile(problems_csv) and os.path.isfile(answers_csv):
            csv_problems = load_csv_problems_by_ids(problems_csv, answers_csv, csv_ids)
        else:
            csv_problems = []
            print(f'WARNING: CSV not found at {problems_csv}, using tier 0 only')

        tier1_problems = csv_problems
        print(f'\nTier 1: {len(tier1_problems)} problems (previous failures)')

        c1, t1, r1 = run_test_batch('TIER 1 — PREVIOUS FAILURES', tier1_problems, solver)
        total_correct += c1
        total_problems += t1
        all_results.extend(r1)
    else:
        print(f'\nSKIPPING Tier 1 (TEST_TIER={TEST_TIER})')

    # --- TIER 2: Full val bench (remaining problems not in tier 0/1) ---
    if TEST_TIER >= 2:
        if os.path.isfile(problems_csv) and os.path.isfile(answers_csv):
            val_problems = load_csv_problems(problems_csv, answers_csv)
            used_ids = set(p['id'] for p in tier0_problems)
            if TEST_TIER >= 1:
                used_ids |= set(V31_WRONG_IDS) | set(DODGY_WIN_IDS) | set(TIER_15_IDS)
            val_problems = [p for p in val_problems if p['id'] not in used_ids]
            MAX_TIER2 = 35
            if len(val_problems) > MAX_TIER2:
                import random
                random.seed(42)
                val_problems = random.sample(val_problems, MAX_TIER2)
                print(f'  Tier 2 capped at {MAX_TIER2} (from {len(val_problems)+MAX_TIER2-MAX_TIER2})')
            c, t, r = run_test_batch('TIER 2 — VAL BENCH (remaining)', val_problems, solver)
            total_correct += c
            total_problems += t
            all_results.extend(r)
        else:
            print(f'SKIPPING Tier 2: CSV not found')
    else:
        print(f'\nSKIPPING Tier 2 (TEST_TIER={TEST_TIER})')

    # =========================================================================
    #  GPU METRICS SUMMARY
    # =========================================================================
    try:
        solver.print_gpu_summary()
    except Exception as e:
        print(f'GPU metrics unavailable: {e}')

    # =========================================================================
    #  FINAL SUMMARY
    # =========================================================================
    suite_elapsed = time.time() - suite_start
    print(f'\n{"="*80}')
    print(f'  FINAL SUMMARY')
    print(f'{"="*80}')
    print(f'  Score: {total_correct}/{total_problems} ({total_correct/max(total_problems,1)*100:.1f}%)')
    print(f'  Total time: {suite_elapsed:.0f}s ({suite_elapsed/60:.1f} min)')
    print(f'  TEST_TIER: {TEST_TIER}')

    if all_results:
        print(f'\n  {"ID":<8} {"Status":<8} {"Pred":>8} {"Exp":>8} {"Votes":<24} {"Ans":<6} {"Errs":<5} {"Time":>6}')
        print(f'  {"---":<8} {"---":<8} {"---":>8} {"---":>8} {"---":<24} {"---":<6} {"---":<5} {"---":>6}')
        for r in all_results:
            status = 'OK' if r['correct'] else 'FAIL'
            votes_str = str(r.get('votes', {}))
            if len(votes_str) > 22:
                votes_str = votes_str[:22] + '..'
            n_ans = r.get('num_answers', '?')
            n_err = r.get('num_errors', '?')
            print(f'  {r["id"]:<8} {status:<8} {r["predicted"]:>8} {r["expected"]:>8} {votes_str:<24} {str(n_ans):<6} {str(n_err):<5} {r["time"]:>5.0f}s')

        wrong_results = [r for r in all_results if not r['correct']]
        correct_results = [r for r in all_results if r['correct']]
        if wrong_results:
            print(f'\n  WRONG ANSWERS ({len(wrong_results)}):')
            for r in wrong_results:
                print(f'    {r["id"]}: predicted {r["predicted"]}, expected {r["expected"]}, votes={r.get("votes", "?")}')
        if correct_results:
            avg_time = sum(r['time'] for r in correct_results) / len(correct_results)
            print(f'\n  Correct answers avg time: {avg_time:.0f}s')

    print(f'\n{"="*80}')
